### This notebook calculates the imapct chains using the BD_RiesgoClimatico_IKI sql DB

**Created:** 12/09/2025 by Sophia Bakar (sbakar@rti.org)

**Project #:** 0219481  

**Last modified:** 12/26/2025 by Sophia Bakar
 
**Status:** in progress

**QA Status:** reviewed by  

**Original Script Stored at:** Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\

**Objective:**   

**Compatibility:** 

**Packages:** numpy, pandas, geopandas, sqlite3, matplotlib ...  

**Further documentation:**  
 
**Inputs:**   

**Outputs:** 
 
**Assumptions:** Assumes that the min and max values listed in the Indicators table of the database is correct/appropriate for normalizing the indicator values using a min/max aproach.
 
**Future work:** 
 
**Notes:** This script queries the IKI Climate Risk SQL database to generate the impact chains for the desired set of user's weights and selected scenario. Prior to calculating the impact chain, we normalize the values using a min/max approach. The resulting impact chain values will be populated into a new table "ImpactChain_Results".

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import sqlite3
import os

In [2]:
# Connect to the SQLite database
# user = 'sgilson'
#db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"
conn = sqlite3.connect(db_path)

In [3]:
# Create Impact Chain Results table 

## should the names be in English?

cursor = conn.cursor()

# cursor.execute("DROP TABLE IF EXISTS ImpactChain_Results;") # uncomment to reset table/make changes to table structure

cursor.execute("""
CREATE TABLE IF NOT EXISTS ImpactChain_Results (
    ResultID INTEGER PRIMARY KEY AUTOINCREMENT,
    ScnID INTEGER,
    IcID INTEGER,
    UserID INTEGER,
    COMID INTEGER,
    Peligro REAL,
    Exposicion REAL,
    VSS REAL,
    VSB REAL,
    VCA REAL,
    Vulnerabilidad REAL,
    Riesgo REAL,
    UNIQUE (ScnID, IcID, UserID, COMID)
);
""")
conn.commit()


In [4]:
# function/sql query to get indicator data and weights for each impact chain and scenario

def get_indicator_data(scn_id, ic_id, user_id):
    """
    Returns indicator values, weights, and min/max bounds.
    """
    query = """
        SELECT
            ici.IndID,
            iv.COMID,
            iv.Value,
            iw.Factor,
            iw.TextID,
            iw.WeightValue,
            ind.Min AS IndMin,
            ind.Max AS IndMax
        FROM ImpactChain_Indicators ici
        JOIN IndValues_Dyn iv
            ON ici.IndID = iv.IndID
        JOIN IndicatorWeights iw
            ON ici.IcID = iw.IcID
           AND ici.IndID = iw.IndID
           AND iw.UserID = ?
        JOIN Indicators ind
            ON ici.IndID = ind.IndID
        WHERE ici.IcID = ?
          AND iv.ScnID = ?;
    """
    return pd.read_sql(query, conn, params=(user_id, ic_id, scn_id))


In [5]:
# helper function to perform QC checks on indicator ranges for min/max values
def qc_check_indicator_ranges(df):
    """
    Print observed vs table min/max for QC.
    """
    print("\nQC CHECK: Indicator ranges")
    for ind_id, g in df.groupby("IndID"):
        obs_min = g["Value"].min()
        obs_max = g["Value"].max()
        tbl_min = g["IndMin"].iloc[0]
        tbl_max = g["IndMax"].iloc[0]

        print(
            f"IndID {ind_id}: "
            f"Observed min/max = ({obs_min:.3f}, {obs_max:.3f}) | "
            f"Table min/max = ({tbl_min:.3f}, {tbl_max:.3f})"
        )


In [6]:
# function to perform the min/max normalization
def min_max_normalize(df):
    """
    Apply min–max scaling using Indicators.Min and Indicators.Max.
    """
    df = df.copy()

    def scale(row):
        if pd.isna(row["Value"]):
            return np.nan
        denom = row["IndMax"] - row["IndMin"]
        if denom == 0:
            return 0.0
        return (row["Value"] - row["IndMin"]) / denom

    df["ValueNorm"] = df.apply(scale, axis=1)
    return df

In [7]:
# compute weighted index for Peligro and Exposición Indicators
def compute_weighted_index(df, factor_name):
    """
    Compute weighted average for factor 'P' or 'E'.

    Rules:
    - Value == 0  → weight included
    - Value == NaN → weight excluded
    """
    df_factor = df[df["Factor"] == factor_name].copy()
    if df_factor.empty:
        return 0.0

    vals = pd.to_numeric(df_factor["Value"], errors="coerce")
    ws   = pd.to_numeric(df_factor["WeightValue"], errors="coerce")

    # keep only rows with non-NaN values
    mask = ~vals.isna()
    if not mask.any():
        return 0.0

    vals = vals[mask]
    ws   = ws[mask]

    denom = ws.sum()
    if denom == 0:
        return 0.0

    return float((vals * ws).sum() / denom)

In [8]:
# compute weighted index for Vulnerability Indicators

def compute_vulnerability(df):
    """
    Compute Vulnerability:

    Vulnerabilidad =
      (VSB * W_vsb + VSS * W_vss - VCA * W_vca)
      / (W_vsb + W_vss + W_vca)

    Rules:
    - Indicator value == 0  → weight included
    - Indicator value == NaN → weight excluded
    """
    df_v = df[df["Factor"] == "V"].copy()
    
    if df_v.empty:
        return 0.0, 0.0, 0.0, 0.0

    categories = ["VSB", "VSS", "VCA"]
    results = {}

    for cat in categories:
        sub = df_v[df_v["TextID"].str.strip().str.upper().str.startswith(cat)].copy()
        if sub.empty:
            results[cat] = {"avg": 0.0, "w_sum": 0.0}
            continue

        vals = pd.to_numeric(sub["Value"], errors="coerce")
        ws   = pd.to_numeric(sub["WeightValue"], errors="coerce")

        mask = ~vals.isna()
        if not mask.any():
            results[cat] = {"avg": 0.0, "w_sum": 0.0}
            continue

        vals = vals[mask]
        ws   = ws[mask]

        w_sum = ws.sum()
        avg = (vals * ws).sum() / w_sum if w_sum != 0 else 0.0

        results[cat] = {"avg": float(avg), "w_sum": float(w_sum)}

    # unpack
    VSB, w_vsb = results["VSB"]["avg"], results["VSB"]["w_sum"]
    VSS, w_vss = results["VSS"]["avg"], results["VSS"]["w_sum"]
    VCA, w_vca = results["VCA"]["avg"], results["VCA"]["w_sum"]

    denom = w_vsb + w_vss + w_vca
    if denom == 0:
        return 0.0, VSS, VSB, VCA

    vulnerabilidad = (
        VSB * w_vsb +
        VSS * w_vss -
        VCA * w_vca
    ) / denom

    return float(vulnerabilidad), VSS, VSB, VCA

In [9]:
def get_factor_weights(ic_id, user_id):
    """
    Returns a dict with keys 'P','V','E' mapping to weight values.
    Expects FactorWeights to have Factor column with values 'P','V','E'.
    """
    q = """
        SELECT Factor, WeightValue
        FROM FactorWeights
        WHERE IcID = ? AND UserID = ?;
    """
    df = pd.read_sql(q, conn, params=(ic_id, user_id))

    # No user-defined weights at all → equal weights
    if df.empty:
        return {"P": 1/3, "V": 1/3, "E": 1/3}

    # Partial user-defined weights → fill missing with 1/3
    weights = {
        row["Factor"]: float(row["WeightValue"])
        for _, row in df.iterrows()
    }

    return {
        "P": weights.get("P", 1/3),
        "V": weights.get("V", 1/3),
        "E": weights.get("E", 1/3)
    }


In [10]:
# replace values based on User, Scenario, COMID, and Impact Chain
def insert_results(
    scn_id, ic_id, user_id, comid,
    p, e, vss, vsb, vca, vulnerabilidad, r
):
    q = """
        INSERT OR REPLACE INTO ImpactChain_Results
        (ScnID, IcID, UserID, COMID,
         Peligro, Exposicion, VSS, VSB, VCA,
         Vulnerabilidad, Riesgo)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """
    conn.execute(q, (
        scn_id, ic_id, user_id, comid,
        p, e, vss, vsb, vca, vulnerabilidad, r
    ))
    conn.commit()

In [11]:
# function to process an impact chain for a given scenario and user; 
def process_ic_scenario(scn_id, ic_id, user_id, run_qc):
    # Get indicator data (includes Min/Max)
    df = get_indicator_data(scn_id, ic_id, user_id)
    if df.empty:
        print(f"No data for IcID={ic_id}, ScnID={scn_id}, UserID={user_id}")
        return

    # Optional QC check
    if run_qc:
        qc_check_indicator_ranges(df)

    # Apply min-max normalization
    df = min_max_normalize(df)
    # Use normalized values for all calculations
    df["Value"] = df["ValueNorm"]

    # Get user-specific factor weights
    factor_weights = get_factor_weights(ic_id, user_id)

    # Group by COMID
    for comid, g in df.groupby("COMID"):
        # Compute Peligro and Exposición (weighted average)
        p = compute_weighted_index(g, "P")
        e = compute_weighted_index(g, "E")
        # Compute Vulnerabilidad (weighted avg of VSB, VSS, VCA)
        v, vss, vsb, vca = compute_vulnerability(g)

        # Aggregate Risk using factor weights
        fw = factor_weights
        denom = fw["P"] + fw["V"] + fw["E"]
        r = (p * fw["P"] + v * fw["V"] + e * fw["E"]) / denom if denom != 0 else np.mean([p, v, e])
        # Optional: truncate negative risk to zero if desired
        # r = max(0, r)

        # Insert results into DB
        insert_results(
            scn_id, ic_id, user_id, comid,
            p, e, vss, vsb, vca, v, r
        )

In [12]:
# to run subset of impact chains
if __name__ == "__main__":
    conn = sqlite3.connect(db_path)
    user_id = 1                 # select user
    scenarios = [1]             # update as we add more scenarios
    impact_chains = [11]         # set impact chain ID(s)
    run_qc = True               # set run_qc = True to perform QC check on min and max, otherwise set False

    for scn in scenarios:
        for ic in impact_chains:
            print(f"Processing: ScnID={scn}, IcID={ic}, UserID={user_id}")
            process_ic_scenario(
                scn_id=scn,
                ic_id=ic,
                user_id=user_id,
                run_qc=run_qc
            )

    conn.close()

Processing: ScnID=1, IcID=11, UserID=1

QC CHECK: Indicator ranges
IndID 102: Observed min/max = (1.000, 4.000) | Table min/max = (1.000, 4.000)
IndID 107: Observed min/max = (1.000, 4.000) | Table min/max = (1.000, 4.000)
IndID 207: Observed min/max = (0.000, 599025.000) | Table min/max = (0.000, 700000.000)
IndID 408: Observed min/max = (0.000, 1.000) | Table min/max = (0.000, 1.000)
IndID 504: Observed min/max = (0.000, 1.000) | Table min/max = (0.000, 1.000)
IndID 515: Observed min/max = (0.001, 7.971) | Table min/max = (0.000, 10.000)


In [ ]:
# to run all impact chains

if __name__ == "__main__":
    user_id = 1                 # select user
    scenarios = [1]             # update based on scenario
    run_qc = False              # set run_qc = True to perform QC check on min and max, otherwise set False

    impact_chains = pd.read_sql(
        "SELECT DISTINCT IcID FROM ImpactChain_Indicators;",
        conn
    )["IcID"].tolist()

    for scn in scenarios:
        for ic in impact_chains:
            print(f"Processing: ScnID={scn}, IcID={ic}, UserID={user_id}")
            process_ic_scenario(
                scn_id=scn,
                ic_id=ic,
                user_id=user_id,
                run_qc=run_qc
            )

    conn.close()